# Phase 6 - Architecture 4 (encoder-decoder) (PEFT: LoRMA by default)

Abstractive legal summarization, starting from the Phase 1 legal language model and the Phase 1 tokenizer (both unchanged). The base model is frozen and only a small set of parameters from `q4_peft` is trained. Set `METHOD` to `soft_prompt`, `adapter`, `lora` or `lorma` (all four q4_peft methods are implemented). No knowledge distillation is used.

**Long documents:** a document longer than the input window keeps its first 60% and last 40% (the middle is dropped; change `HEAD_FRACTION` to try other splits).

**Before running:** attach the `kratikasum/tokenizer` dataset and your Phase 1 checkpoint dataset, add the Kaggle secret `HF`, and set `CKPT` in the settings cell.

**Encoder-decoder:** the encoder reads exactly 768 document tokens (its Phase 1 `enc_len`), the decoder writes the summary.

## 1. Setup

### Imports, Hugging Face login, GPU check

In [26]:
import os, random, json, time, math, zipfile, glob, pickle, re
from typing import List, Dict, Tuple
import numpy as np
import torch
from huggingface_hub import login, hf_hub_download
from kaggle_secrets import UserSecretsClient

# Log in to Hugging Face (the Kaggle secret must be called "HF")
HF_TOKEN = UserSecretsClient().get_secret("HF")
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device, "| GPUs:", torch.cuda.device_count())


# mixed-precision dtype: bf16 on Ampere+ GPUs, fp16 on T4 and older
amp_dtype = torch.bfloat16 if (device == "cuda" and torch.cuda.get_device_capability()[0] >= 8) else torch.float16

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


device: cuda | GPUs: 2


### Install metric libraries

In [27]:
# Metric libraries (BERTScore is the primary metric, ROUGE the secondary ones)
!pip install -q rouge_score


### Settings (the only cell you normally need to edit)

In [28]:
# ---------------- which Phase 1 model and which PEFT method ----------------
ARCH_NAME = "arch4_encdec"
METHOD    = "soft_prompt"        # "soft_prompt" | "adapter" | "lora" | "lorma"   (all four come from q4_peft)

# Phase 1 checkpoint (the pretrained legal language model). Edit this path to your own Kaggle dataset.
CKPT = "/kaggle/input/datasets/kratikasum/model-arch-4/last.pt"

# ---------------- data ----------------
HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
TRAIN_FILE = "Phase_6_Summ/Train/train.json"
TEST_FILE  = "Phase_6_Summ/Test/test.json"
VAL_FRACTION = 0.05             # no public dev set, so we hold out 5% of train

# ---------------- sequence lengths (in tokens) ----------------
DOC_LEN = 768                 # how many document tokens the model reads
HEAD_FRACTION = 0.6             # long documents: share of the window given to the START of the document (rest = END)
                                # 1.0 = head only. Re-run with another value to compare long-document strategies.
SUM_LEN = 256                   # max summary tokens (training target and generation limit)
SEP_IDS_MODE = "none"        # "bos" = put a <bos> token between document and summary (decoder-only models)
                                # "none" = no separator (encoder-decoder: encoder reads doc, decoder writes summary)

# ---------------- PEFT hyper-parameters ----------------
RANK, ALPHA = 8, 16             # LoRA / LoRMA rank and alpha (scaling = alpha / rank = 2)
BOTTLENECK  = 64                # adapter bottleneck size
SOFT_TOKENS = 16                # soft prompt: number of trainable "virtual tokens" placed in front of the document
SOFT_INIT_TEXT = " Summarize the following legal judgment:"   # soft prompt starts from the embeddings of these words
LR = {"soft_prompt": 3e-3, "adapter": 5e-4, "lora": 3e-4, "lorma": 2e-4}[METHOD]

# ---------------- training ----------------
MICRO_BATCH = 4                # sequences per forward pass (lower it if you get CUDA out-of-memory)
GRAD_ACCUM  = 4                # effective batch = MICRO_BATCH * GRAD_ACCUM
EPOCHS = 2
WARMUP_STEPS = 30
WEIGHT_DECAY = 0.01
GRAD_CLIP = 1.0
LOG_EVERY = 20
EVAL_EVERY = 200
MAX_TRAIN_MINUTES = 540         # safety cap so the notebook always finishes inside the Kaggle session

# ---------------- generation ----------------
GEN_BATCH = 16
REPETITION_PENALTY = 1.15       # discourages repeating tokens already generated
NO_REPEAT_NGRAM = 3             # forbids repeating any 3-token phrase
N_VAL_GEN = 100                 # how many held-out documents to score with BERTScore / ROUGE
N_BASE_GEN = 30                 # documents used to score the Phase 1 model BEFORE fine-tuning (for the analysis)

OUTPUT_DIR = f"/kaggle/working/phase6_{ARCH_NAME}_{METHOD}"
os.makedirs(OUTPUT_DIR, exist_ok=True)
PEFT_WEIGHTS_PATH = os.path.join(OUTPUT_DIR, "peft_weights.pt")
PREDICTION_PATH   = os.path.join(OUTPUT_DIR, "prediction.json")
SUBMISSION_ZIP    = os.path.join(OUTPUT_DIR, "submission.zip")


## 2. Tokenizer (Phase 1, unchanged)

### Tokenizer class

In [29]:
# This is the Phase 1 tokenizer class, unchanged (training code removed, we only load the pickle).
class ByteLevelBPE:
    def __init__(self):
        self.vocab: Dict[bytes, int] = {}
        self.inverse_vocab: Dict[int, bytes] = {}
        self.merges: Dict[Tuple[bytes, bytes], Tuple[bytes, int]] = {}
        self.word_end_marker = b'</w>'
        self.reserved_tokens = [b'<pad>', b'<unk>', b'<bos>', b'<eos>']

    def tokenize(self, text: str) -> List[bytes]:
        words = text.split()
        tokens = []
        for w in words:
            word_bytes = w.encode('utf-8')
            split = [bytes([b]) for b in word_bytes] + [self.word_end_marker]
            while True:
                best_pair, best_rank = None, float('inf')
                for i in range(len(split) - 1):
                    pair = (split[i], split[i+1])
                    if pair in self.merges:
                        rank = self.merges[pair][1]
                        if rank < best_rank:
                            best_rank, best_pair = rank, pair
                if best_pair is None:
                    break
                new_split, i = [], 0
                while i < len(split):
                    if i < len(split) - 1 and split[i] == best_pair[0] and split[i+1] == best_pair[1]:
                        new_split.append(self.merges[best_pair][0])
                        i += 2
                    else:
                        new_split.append(split[i])
                        i += 1
                split = new_split
            tokens.extend(split)
        return tokens

    def encode(self, text: str) -> List[int]:
        unk_id = self.vocab[b'<unk>']
        return [self.vocab.get(token, unk_id) for token in self.tokenize(text)]

    def decode(self, token_ids: List[int]) -> str:
        skip_ids = {self.vocab[t] for t in self.reserved_tokens if t in self.vocab}
        bytes_list = []
        for tid in token_ids:
            if tid in skip_ids:
                continue
            bytes_list.append(self.inverse_vocab.get(tid, b''))
        joined = b''.join(bytes_list)
        return joined.replace(self.word_end_marker, b' ').decode('utf-8', errors='replace').strip()

    @classmethod
    def load(cls, filepath: str):
        with open(filepath, 'rb') as f:
            data = pickle.load(f)
        bpe = cls()
        bpe.vocab = data['vocab']
        bpe.inverse_vocab = {v: k for k, v in bpe.vocab.items()}
        bpe.merges = data['merges']
        return bpe

    def __len__(self):
        return len(self.vocab)

    @property
    def eos_token_id(self):
        return self.vocab.get(b'<eos>')


### Load the trained tokenizer

In [30]:
# Find the tokenizer pickle in the attached Kaggle dataset (kratikasum/tokenizer)
cands = glob.glob("/kaggle/input/**/bpe_tokenized_txt.pkl", recursive=True)
tok_path = cands[0]
tokenizer = ByteLevelBPE.load(tok_path)

VOCAB_SIZE = max(tokenizer.vocab.values()) + 1
PAD = tokenizer.vocab[b"<pad>"]
BOS = tokenizer.vocab[b"<bos>"]
EOS = tokenizer.vocab[b"<eos>"]
print("vocab size:", VOCAB_SIZE, "| PAD/BOS/EOS ids:", PAD, BOS, EOS)

# The tokenizer works word by word, so we cache the result for every word (much faster, same output)
word_cache = {}
def fast_encode(text):
    out = []
    for word in text.split():
        if word not in word_cache:
            word_cache[word] = tokenizer.encode(word)
        out.extend(word_cache[word])
    return out


vocab size: 48000 | PAD/BOS/EOS ids: 0 2 3


## 3. Phase 6 data

In [31]:
# Download Phase 6 data. Train items look like {"id", "document", "summary"}, test items have no summary.
train_path = hf_hub_download(repo_id=HF_DATASET_REPO, filename=TRAIN_FILE, repo_type="dataset", token=HF_TOKEN)
test_path  = hf_hub_download(repo_id=HF_DATASET_REPO, filename=TEST_FILE,  repo_type="dataset", token=HF_TOKEN)
train_all = json.load(open(train_path, encoding="utf-8"))
test_docs = json.load(open(test_path, encoding="utf-8"))

# Hold out a small validation split from the training data
random.Random(SEED).shuffle(train_all)
n_val = max(1, int(len(train_all) * VAL_FRACTION))
val_docs, train_docs = train_all[:n_val], train_all[n_val:]
print(f"train: {len(train_docs)} | validation: {len(val_docs)} | test: {len(test_docs)}")


train: 6679 | validation: 351 | test: 100


## 4. Phase 1 model code (unchanged)

### Imports and config

In [32]:
import math
from dataclasses import dataclass
from types import SimpleNamespace
from typing import List, Optional
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint


@dataclass
class Config:
    vocab_size: int = 32000
    d_model: int = 768
    n_head: int = 12
    n_kv_head: Optional[int] = None
    max_seq_len: int = 2048
    rope_theta: float = 10000.0
    rope_pct: float = 1.0
    dropout: float = 0.0
    tie_embeddings: bool = False
    qk_norm: bool = False
    grad_ckpt: bool = False
    n_enc_layer: int = 6
    n_dec_layer: int = 6
    enc_len: int = 768                   # tokens given to the encoder; the rest is the decoder target
    cross_every: int = 1
    enc_downsample: int = 1
    mtp_heads: int = 0
    mtp_weight: float = 0.3

    @property
    def head_dim(self):
        return self.d_model // self.n_head

    @property
    def kv_heads(self):
        return self.n_kv_head or self.n_head

    @property
    def ff_dim(self):
        return int(round(8 / 3 * self.d_model / 64)) * 64


### Building blocks

In [33]:
# ---- building blocks of the Phase 1 models (identical to the pretraining notebooks) ----
class RMSNorm(nn.Module):
    def __init__(self, d, eps=1e-6):
        super().__init__()
        self.w = nn.Parameter(torch.ones(d))
        self.eps = eps

    def forward(self, x):
        xf = x.float()
        out = xf * torch.rsqrt(xf.pow(2).mean(-1, keepdim=True) + self.eps)
        return (out * self.w.float()).type_as(x)


class Rotary(nn.Module):
    def __init__(self, dim, max_len, theta):
        super().__init__()
        inv = 1.0 / (theta ** (torch.arange(0, dim, 2).float() / max(dim, 1)))
        freqs = torch.outer(torch.arange(max_len).float(), inv)
        self.register_buffer("cos", freqs.cos(), persistent=False)
        self.register_buffer("sin", freqs.sin(), persistent=False)

    def forward(self, T, start=0):
        return self.cos[start:start + T], self.sin[start:start + T]


def apply_rope(x, cos, sin, rot_dim):
    if rot_dim == 0:
        return x
    xr, xp = x[..., :rot_dim], x[..., rot_dim:]
    x1, x2 = xr.float().chunk(2, dim=-1)
    c, s = cos[None, None], sin[None, None]
    xr = torch.cat([x1 * c - x2 * s, x1 * s + x2 * c], dim=-1).to(x.dtype)
    return torch.cat([xr, xp], dim=-1)


class MLP(nn.Module):
    """SwiGLU feed-forward."""
    def __init__(self, cfg):
        super().__init__()
        d, f = cfg.d_model, cfg.ff_dim
        self.gate = nn.Linear(d, f, bias=False)
        self.up = nn.Linear(d, f, bias=False)
        self.down = nn.Linear(f, d, bias=False)

    def forward(self, x):
        return self.down(F.silu(self.gate(x)) * self.up(x))


class KVCache:
    """Key/value buffer for one attention layer, used for fast generation."""
    def __init__(self, max_len):
        self.max_len, self.n, self.k, self.v = max_len, 0, None, None

    def append(self, k, v):
        if self.k is None:
            B, H, _, hd = k.shape
            self.k = k.new_empty(B, H, self.max_len, hd)
            self.v = v.new_empty(B, H, self.max_len, hd)
        T = k.size(2)
        self.k[:, :, self.n:self.n + T] = k
        self.v[:, :, self.n:self.n + T] = v
        self.n += T

    def get(self, window=None):
        s = 0 if window is None else max(0, self.n - window)
        return self.k[:, :, s:self.n], self.v[:, :, s:self.n]


def sdpa_gqa(q, k, v, mask=None, causal=False, p=0.0):
    B, H, T, hd = q.shape
    Hkv = k.size(1)
    if T == 1 and mask is None:
        if Hkv != H:
            r = H // Hkv
            out = F.scaled_dot_product_attention(q.reshape(B, Hkv, r, hd), k, v)
            return out.reshape(B, H, 1, hd)
        return F.scaled_dot_product_attention(q, k, v)
    if Hkv != H:
        r = H // Hkv
        k, v = k.repeat_interleave(r, dim=1), v.repeat_interleave(r, dim=1)
    return F.scaled_dot_product_attention(q, k, v, attn_mask=mask, dropout_p=p,
                                          is_causal=causal and mask is None)


class Attention(nn.Module):
    """MHA/GQA self-attention. Its projections are called q, k, v, o (PEFT targets these names)."""
    def __init__(self, cfg, causal=True, use_rope=True, value_residual=False):
        super().__init__()
        d = cfg.d_model
        self.H, self.Hkv, self.hd = cfg.n_head, cfg.kv_heads, cfg.head_dim
        assert self.H % self.Hkv == 0, "n_head must be divisible by n_kv_head"
        self.q = nn.Linear(d, self.H * self.hd, bias=False)
        self.k = nn.Linear(d, self.Hkv * self.hd, bias=False)
        self.v = nn.Linear(d, self.Hkv * self.hd, bias=False)
        self.o = nn.Linear(self.H * self.hd, d, bias=False)
        self.q_norm = RMSNorm(self.hd) if cfg.qk_norm else None
        self.k_norm = RMSNorm(self.hd) if cfg.qk_norm else None
        self.p_drop = cfg.dropout
        self.causal = causal
        self.rot_dim = (int(self.hd * cfg.rope_pct) // 2 * 2) if use_rope else 0
        self.lam = nn.Parameter(torch.zeros(self.Hkv, 1, 1)) if value_residual else None

    def forward(self, x, rope=None, v_first=None, mask=None, cache=None, window=None):
        B, T, _ = x.shape
        q = self.q(x).view(B, T, self.H, self.hd).transpose(1, 2)
        k = self.k(x).view(B, T, self.Hkv, self.hd).transpose(1, 2)
        v = self.v(x).view(B, T, self.Hkv, self.hd).transpose(1, 2)
        if self.lam is not None and v_first is not None:
            lam = torch.sigmoid(self.lam).to(v.dtype)
            v = (1 - lam) * v + lam * v_first
        v_new = v
        if self.q_norm is not None:
            q, k = self.q_norm(q), self.k_norm(k)
        if self.rot_dim:
            cos, sin = rope
            q, k = apply_rope(q, cos, sin, self.rot_dim), apply_rope(k, cos, sin, self.rot_dim)
        causal = self.causal
        if cache is not None:
            cache.append(k, v)
            k, v = cache.get(window if T == 1 else None)
            causal = T > 1
        out = sdpa_gqa(q, k, v, mask=mask, causal=causal, p=self.p_drop if self.training else 0.0)
        return self.o(out.transpose(1, 2).reshape(B, T, -1)), v_new


class LMHead(nn.Module):
    """Final norm + unembedding (+ multi-token-prediction heads used only for the training loss)."""
    def __init__(self, cfg, tok_emb: nn.Embedding):
        super().__init__()
        self.cfg = cfg
        self.norm = RMSNorm(cfg.d_model)
        self.proj = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        if cfg.tie_embeddings:
            self.proj.weight = tok_emb.weight
        self.mtp = nn.ModuleList([nn.Linear(cfg.d_model, cfg.d_model, bias=False)
                                  for _ in range(getattr(cfg, "mtp_heads", 0))])

    def logits(self, h):
        return self.proj(self.norm(h))

    def forward(self, h, targets=None):
        h = self.norm(h)
        logits = self.proj(h)
        if targets is None:
            return logits, None
        V = logits.size(-1)
        loss = F.cross_entropy(logits.reshape(-1, V).float(), targets.reshape(-1), ignore_index=-100)
        self.main_loss = loss.detach()          # plain next-token loss, used for logging
        w = getattr(self.cfg, "mtp_weight", 0.3)
        for k, p in enumerate(self.mtp, start=1):
            if targets.size(1) <= k:
                break
            hk = h[:, :-k] + F.silu(p(h[:, :-k]))
            lk = self.proj(hk)
            loss = loss + w * F.cross_entropy(lk.reshape(-1, V).float(), targets[:, k:].reshape(-1),
                                              ignore_index=-100)
        return logits, loss


def ckpt_call(fn, enabled, *args):
    return checkpoint(fn, *args, use_reentrant=False) if enabled else fn(*args)


def init_weights(model, depth):
    for m in model.modules():
        if isinstance(m, (nn.Linear, nn.Embedding)):
            nn.init.normal_(m.weight, 0.0, 0.02)
    std = 0.02 / math.sqrt(2 * depth)
    for n, p in model.named_parameters():
        if n.endswith(("o.weight", "down.weight")):
            nn.init.normal_(p, 0.0, std)


class BaseLM(nn.Module):
    eval_offset = 0

    def num_params(self, non_embedding=False):
        n = sum(p.numel() for p in self.parameters())
        return n - self.tok.weight.numel() if non_embedding else n


### Architecture

In [34]:
class CrossAttention(nn.Module):
    """Decoder -> encoder attention (projections are called q, k, v, o as well)."""
    def __init__(self, cfg: Config):
        super().__init__()
        d = cfg.d_model
        self.H, self.Hkv, self.hd = cfg.n_head, cfg.kv_heads, cfg.head_dim
        assert self.H % self.Hkv == 0, "n_head must be divisible by n_kv_head"
        self.q = nn.Linear(d, self.H * self.hd, bias=False)
        self.k = nn.Linear(d, self.Hkv * self.hd, bias=False)
        self.v = nn.Linear(d, self.Hkv * self.hd, bias=False)
        self.o = nn.Linear(self.H * self.hd, d, bias=False)
        self.q_norm = RMSNorm(self.hd) if cfg.qk_norm else None
        self.k_norm = RMSNorm(self.hd) if cfg.qk_norm else None
        self.p_drop = cfg.dropout

    def forward(self, x, mem, slot=None):
        B, T, _ = x.shape
        q = self.q(x).view(B, T, self.H, self.hd).transpose(1, 2)
        if slot is not None and slot.k is not None:
            k, v = slot.k, slot.v
            if self.q_norm is not None:
                q = self.q_norm(q)
        else:
            S = mem.shape[1]
            k = self.k(mem).view(B, S, self.Hkv, self.hd).transpose(1, 2)
            v = self.v(mem).view(B, S, self.Hkv, self.hd).transpose(1, 2)
            if self.q_norm is not None:
                q, k = self.q_norm(q), self.k_norm(k)
            if slot is not None:
                slot.k, slot.v = k, v
        out = sdpa_gqa(q, k, v, p=self.p_drop if self.training else 0.0)
        return self.o(out.transpose(1, 2).reshape(B, T, -1))


class Block(nn.Module):
    """Bidirectional encoder block."""
    def __init__(self, cfg: Config):
        super().__init__()
        self.n1, self.n2 = RMSNorm(cfg.d_model), RMSNorm(cfg.d_model)
        self.attn, self.mlp = Attention(cfg, causal=False), MLP(cfg)
        self.drop = nn.Dropout(cfg.dropout)

    def forward(self, x, rope):
        h, _ = self.attn(self.n1(x), rope)
        x = x + self.drop(h)
        return x + self.drop(self.mlp(self.n2(x)))


class DecBlock(nn.Module):
    def __init__(self, cfg: Config, cross: bool):
        super().__init__()
        self.n1, self.n2 = RMSNorm(cfg.d_model), RMSNorm(cfg.d_model)
        self.self_attn = Attention(cfg, causal=True)
        self.cross = CrossAttention(cfg) if cross else None
        self.nx = RMSNorm(cfg.d_model) if cross else None
        self.mlp = MLP(cfg)
        self.drop = nn.Dropout(cfg.dropout)

    def forward(self, x, rope, mem, cache=None):
        h, _ = self.self_attn(self.n1(x), rope, cache=None if cache is None else cache.self_kv)
        x = x + self.drop(h)
        if self.cross is not None:
            x = x + self.drop(self.cross(self.nx(x), mem, None if cache is None else cache.cross_kv))
        return x + self.drop(self.mlp(self.n2(x)))


class EncDecBase(BaseLM):
    def __init__(self, cfg: Config):
        super().__init__()
        self.cfg = cfg
        d = cfg.d_model
        self.eval_offset = cfg.enc_len - 1
        self.tok = nn.Embedding(cfg.vocab_size, d)
        self.rope = Rotary(int(cfg.head_dim * cfg.rope_pct) // 2 * 2, cfg.max_seq_len, cfg.rope_theta)
        self.enc_blocks = nn.ModuleList([Block(cfg) for _ in range(cfg.n_enc_layer)])
        self.enc_norm = RMSNorm(d)
        ds = cfg.enc_downsample
        self.down = nn.Conv1d(d, d, kernel_size=ds, stride=ds) if ds > 1 else None
        self.dec_blocks = nn.ModuleList(
            [DecBlock(cfg, cross=(i % cfg.cross_every == cfg.cross_every - 1)) for i in range(cfg.n_dec_layer)])
        self.head = LMHead(cfg, self.tok)
        init_weights(self, cfg.n_enc_layer + cfg.n_dec_layer)

    def encode(self, ids):
        x = self.tok(ids)
        rope = self.rope(ids.shape[1])
        for blk in self.enc_blocks:
            x = ckpt_call(blk, self.cfg.grad_ckpt and self.training, x, rope)
        x = self.enc_norm(x)
        if self.down is not None:
            ds = self.cfg.enc_downsample
            x = self.down(x[:, : x.shape[1] // ds * ds].transpose(1, 2)).transpose(1, 2)
        return x

    def decode(self, ids, mem, caches=None, start=0):
        x = self.tok(ids)
        rope = self.rope(ids.shape[1], start)
        for i, blk in enumerate(self.dec_blocks):
            if caches is None:
                x = ckpt_call(blk, self.cfg.grad_ckpt and self.training, x, rope, mem)
            else:
                x = blk(x, rope, mem, cache=caches[i])
        return x


class EncDecLM(EncDecBase):
    """Training forward + cached generation."""
    def forward(self, idx, targets=None, src=None):
        P = self.cfg.enc_len
        assert idx.shape[1] > P, f"need sequence longer than enc_len={P}"
        mem = self.encode(idx[:, :P])                        # encoder reads the first P tokens (our document)
        h = self.decode(idx[:, P - 1:], mem)                 # decoder starts from the last document token
        labels = targets[:, P - 1:] if targets is not None else None
        return self.head(h, labels)

    def prefill(self, idx, max_new):
        mem = self.encode(idx[:, -self.cfg.enc_len:])
        caches = [SimpleNamespace(self_kv=KVCache(max_new + 1), cross_kv=SimpleNamespace(k=None, v=None))
                  for _ in self.dec_blocks]
        state = SimpleNamespace(mem=mem, caches=caches, pos=0)
        return self.step(idx[:, -1:], state), state

    def step(self, tok, state):
        h = self.decode(tok, state.mem, state.caches, start=state.pos)
        state.pos += tok.shape[1]
        return self.head.logits(h[:, -1])


Model = EncDecLM
BLOCK_LISTS = ["enc_blocks", "dec_blocks"]     # where the transformer blocks live (used by the adapter method)


## 5. Load the Phase 1 pretrained weights

In [35]:
# Rebuild the exact Phase 1 architecture from the config stored inside the checkpoint,
# then load the pretrained weights. Nothing about the architecture is changed.
ck = torch.load(CKPT, map_location="cpu", weights_only=False)
cfg = Config(**ck["cfg"])
cfg.grad_ckpt = True                      # only saves memory while training, does not change the model
model = Model(cfg)
model.load_state_dict(ck["model"])
print(f"loaded {CKPT} (Phase 1 step {ck['step']})")
del ck

model.to(device)
total_params = sum(p.numel() for p in model.parameters())
print(f"{ARCH_NAME}: {total_params / 1e6:.1f}M parameters in the base model")


loaded /kaggle/input/datasets/kratikasum/model-arch-4/last.pt (Phase 1 step 1525)
arch4_encdec: 189.4M parameters in the base model


## 6. Parameter-efficient fine-tuning (from q4_peft)

The base model stays frozen. Only the small PEFT parameters are trained.

In [36]:
# The four PEFT methods below (soft prompt, adapter, LoRA, LoRMA) are ported from q4_peft.py. q4_peft was written for GPT-2's Conv1D layers,
# our Phase 1 models use plain nn.Linear, so only the layer type changes, the maths is the same.

# ---- LoRA:  y = W x + (alpha/r) * B(A(x)),  A random, B zero (so training starts exactly at the base model) ----
class LoRALinear(nn.Module):
    def __init__(self, base, rank=8, alpha=16):
        super().__init__()
        self.base = base                                           # frozen original layer
        self.scaling = alpha / rank
        self.lora_A = nn.Parameter(torch.empty(rank, base.in_features))
        self.lora_B = nn.Parameter(torch.zeros(base.out_features, rank))
        nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))

    def forward(self, x):
        low_rank = F.linear(F.linear(x, self.lora_A.to(x.dtype)), self.lora_B.to(x.dtype))
        return self.base(x) + self.scaling * low_rank


# ---- LoRMA_+ (paper eq. 8):  W' = (s*B*A + I) W0   ->   y = h + s * (h A^T) B^T,  with h = W0 x ----
# The low-rank update multiplies on the OUTPUT side of the frozen layer, so A and B live in the d_out space.
class LoRMALinear(nn.Module):
    def __init__(self, base, rank=8, alpha=16):
        super().__init__()
        self.base = base
        self.scaling = alpha / rank
        d_out = base.out_features
        bound = 1.0 / math.sqrt(d_out)
        self.lorma_A = nn.Parameter((torch.rand(rank, d_out) * 2 - 1) * bound)   # [r, d_out] random
        self.lorma_B = nn.Parameter(torch.zeros(d_out, rank))                    # [d_out, r] zero

    def forward(self, x):
        h = self.base(x)                                                          # frozen output W0 x
        update = (h @ self.lorma_A.to(h.dtype).T) @ self.lorma_B.to(h.dtype).T    # two thin matmuls
        return h + self.scaling * update


# ---- Bottleneck adapter:  h' = h + W_up GELU(W_down h),  W_up zero-initialised (identity at step 0) ----
class BottleneckAdapter(nn.Module):
    def __init__(self, hidden_size, bottleneck=64):
        super().__init__()
        self.down = nn.Linear(hidden_size, bottleneck)
        self.up = nn.Linear(bottleneck, hidden_size)
        nn.init.zeros_(self.up.weight)
        nn.init.zeros_(self.up.bias)

    def forward(self, h):
        return h + self.up(F.gelu(self.down(h)))


# ---- Soft prompt:  SOFT_TOKENS trainable vectors that replace the first SOFT_TOKENS input embeddings ----
# (q4_peft prepends them; here the first SOFT_TOKENS positions of every prompt are reserved placeholders, so the
#  sequence length, positions and labels of the Phase 1 model stay exactly as they were)
class SoftPrompt(nn.Module):
    def __init__(self, init_embeddings):                          # [m, d_model]
        super().__init__()
        self.prompt = nn.Parameter(init_embeddings.detach().clone().float())

    def forward(self, x):                                         # x: [B, T, d_model] embeddings
        m = self.prompt.shape[0]
        if x.shape[1] <= m:                                       # a single generation step: nothing to replace
            return x
        prompt = self.prompt.to(x.dtype).unsqueeze(0).expand(x.shape[0], -1, -1)
        return torch.cat([prompt, x[:, m:]], dim=1)


def attach_peft(model, method):
    """Freeze the whole model, then add the trainable PEFT parameters. Returns the trainable parameter names."""
    for p in model.parameters():
        p.requires_grad_(False)                       # everything frozen first

    if method in ("lora", "lorma"):
        Wrapper = LoRALinear if method == "lora" else LoRMALinear
        # find every attention projection called q, k, v or o (works for all four Phase 1 architectures)
        targets = []
        for parent in model.modules():
            for name, child in parent.named_children():
                if isinstance(child, nn.Linear) and name in ("q", "k", "v", "o"):
                    targets.append((parent, name, child))
        for parent, name, child in targets:           # replace after the search so we do not edit while iterating
            setattr(parent, name, Wrapper(child, RANK, ALPHA))
        print(f"{method}: wrapped {len(targets)} attention projections")

    elif method == "soft_prompt":
        # start from the embeddings of a short instruction (like q4_peft's "vocab" init), cycled to SOFT_TOKENS vectors
        init_ids = fast_encode(SOFT_INIT_TEXT)
        init_ids = [init_ids[i % len(init_ids)] for i in range(SOFT_TOKENS)]
        soft = SoftPrompt(model.tok.weight[torch.tensor(init_ids, device=model.tok.weight.device)])
        model.soft = soft
        # a forward pre-hook on the first transformer block (first ENCODER block for Arch 4) swaps in the soft prompt
        first_block = getattr(model, BLOCK_LISTS[0])[0]
        def pre_hook(module, args):
            return (soft(args[0]),) + tuple(args[1:])
        first_block.register_forward_pre_hook(pre_hook)
        print(f"soft prompt: {SOFT_TOKENS} trainable virtual tokens")

    elif method == "adapter":
        # one adapter after every transformer block, applied with a forward hook (the block code stays untouched)
        n = 0
        for list_name in BLOCK_LISTS:
            for blk in getattr(model, list_name):
                adapter = BottleneckAdapter(cfg.d_model, BOTTLENECK)
                blk.add_module("adapter", adapter)    # registered, so it is saved and trained
                def hook(module, inputs, output, adapter=adapter):
                    if isinstance(output, tuple):     # some blocks return (x, extra)
                        return (adapter(output[0]),) + tuple(output[1:])
                    return adapter(output)
                blk.register_forward_hook(hook)
                n += 1
        print(f"adapter: added {n} adapters")
    else:
        raise ValueError(f"unknown method {method}")

    model.to(device)                                  # moves the new PEFT parameters to the GPU
    return [n for n, p in model.named_parameters() if p.requires_grad]


trainable_names = attach_peft(model, METHOD)
n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
n_all = sum(p.numel() for p in model.parameters())
print(f"trainable parameters: {n_trainable / 1e6:.2f}M of {n_all / 1e6:.1f}M ({100 * n_trainable / n_all:.2f}%)")


soft prompt: 16 trainable virtual tokens
trainable parameters: 0.01M of 189.4M (0.01%)


## 7. Prepare training examples

Each example is `[document tokens] [separator] [summary tokens]`. The loss is computed on the summary only.

In [37]:
# ---- turn (document, summary) text into fixed-length token tensors ----
SEP = [BOS] if SEP_IDS_MODE == "bos" else []
N_SOFT = SOFT_TOKENS if METHOD == "soft_prompt" else 0      # placeholder positions reserved for the soft prompt
PROMPT_LEN = DOC_LEN + len(SEP)             # length of everything the model reads before it starts writing
SEQ_LEN = PROMPT_LEN + SUM_LEN              # prompt + summary


def head_tail(ids, n):
    """Long-document handling: if the document is too long, keep the beginning (facts, arguments) and the
    end (final decision / order), and drop the middle. HEAD_FRACTION of the budget goes to the head, the rest to the tail."""
    if len(ids) <= n:
        return ids
    head = int(n * HEAD_FRACTION)
    return ids[:head] + ids[-(n - head):]


def make_prompt(document):
    """[soft-prompt slots] [PAD ... PAD  doc tokens  (BOS)]  -> always exactly PROMPT_LEN tokens.
    Left padding keeps the end of the document right next to where the summary starts."""
    ids = head_tail(fast_encode(document), DOC_LEN - N_SOFT)
    return [PAD] * N_SOFT + [PAD] * (DOC_LEN - N_SOFT - len(ids)) + ids + SEP   # first N_SOFT slots = soft prompt


def make_example(document, summary):
    prompt = make_prompt(document)
    summ = fast_encode(summary)[:SUM_LEN - 1] + [EOS]          # summary + end-of-summary token
    seq = prompt + summ + [PAD] * (SUM_LEN - len(summ))        # right-pad to SEQ_LEN
    x = seq[:-1]                                               # model input
    y = [-100] * (PROMPT_LEN - 1) + summ + [-100] * (SUM_LEN - len(summ))   # loss ONLY on the summary tokens
    return x, y


def build_tensors(docs):
    xs, ys = [], []
    for d in tqdm(docs, desc="tokenizing"):
        x, y = make_example(d["document"], d["summary"])
        xs.append(x); ys.append(y)
    return torch.tensor(xs, dtype=torch.int32), torch.tensor(ys, dtype=torch.int32)


from tqdm.auto import tqdm
train_x, train_y = build_tensors(train_docs)
val_x, val_y = build_tensors(val_docs)
print("train tensor:", tuple(train_x.shape), "| val tensor:", tuple(val_x.shape))


tokenizing:   0%|          | 0/6679 [00:00<?, ?it/s]

tokenizing:   0%|          | 0/351 [00:00<?, ?it/s]

train tensor: (6679, 1023) | val tensor: (351, 1023)


## 8. Generation

The model reads the document once, then writes the summary one token at a time (greedy decoding).

In [38]:
ban_ids = [PAD, tokenizer.vocab[b"<unk>"], BOS]       # never generate these (EOS is allowed: it ends the summary)
MIN_SUM_TOKENS = 40

def block_repeated_ngrams(logits, gen, n):
    """Forbid any token that would repeat an n-token phrase already written."""
    if gen.size(1) < n:
        return
    for b in range(gen.size(0)):
        g = gen[b].tolist()
        prefix = tuple(g[-(n - 1):])
        for i in range(len(g) - n + 1):
            if tuple(g[i:i + n - 1]) == prefix:
                logits[b, g[i + n - 1]] = float("-inf")


@torch.inference_mode()
def generate_batch(prompts):
    """Greedy decoding with a repetition penalty. prompts: LongTensor [B, PROMPT_LEN] on the GPU.
    The model reads the prompt once (prefill), then writes one token per step using its cache."""
    model.eval()
    B = prompts.size(0)
    with torch.autocast(device_type=device, dtype=amp_dtype, enabled=(device == "cuda")):
        logits, state = model.prefill(prompts, SUM_LEN)
    gen = torch.empty(B, 0, dtype=torch.long, device=device)
    finished = torch.zeros(B, dtype=torch.bool, device=device)
    for i in range(SUM_LEN):
        logits = logits.float()
        logits[:, ban_ids] = float("-inf")
        if i < MIN_SUM_TOKENS:                   
            logits[:, EOS] = float("-inf")
        if gen.size(1) > 0:
            # repetition penalty on tokens that were already generated
            s = torch.gather(logits, 1, gen)
            s = torch.where(s < 0, s * REPETITION_PENALTY, s / REPETITION_PENALTY)
            logits = logits.scatter(1, gen, s)
            block_repeated_ngrams(logits, gen, NO_REPEAT_NGRAM)
        nxt = logits.argmax(-1, keepdim=True)
        nxt[finished] = EOS                                   # rows that already ended just keep emitting EOS
        gen = torch.cat([gen, nxt], dim=1)
        finished |= (nxt.squeeze(1) == EOS)
        if bool(finished.all()) or i + 1 == SUM_LEN:
            break
        with torch.autocast(device_type=device, dtype=amp_dtype, enabled=(device == "cuda")):
            logits = model.step(nxt, state)
    model.train()

    summaries = []
    for row in gen.tolist():
        if EOS in row:
            row = row[:row.index(EOS)]                        # cut at the end-of-summary token
        summaries.append(tokenizer.decode(row))
    return summaries


def summarize(docs):
    """List of {"document": ...} -> list of generated summaries (same order)."""
    prompts = torch.tensor([make_prompt(d["document"]) for d in docs], dtype=torch.long)
    out = []
    for i in tqdm(range(0, len(docs), GEN_BATCH), desc="generating"):
        out.extend(generate_batch(prompts[i:i + GEN_BATCH].to(device)))
    return out


### Scoring helper

In [39]:
# Scoring helper used for the baseline, the validation results and the analysis.
# ROUGE-1 / ROUGE-L F1 (no external pretrained model is used for evaluation).
from rouge_score import rouge_scorer

rs = rouge_scorer.RougeScorer(["rouge1", "rougeL"])


def evaluate(preds, refs):
    """Returns the mean scores and the per-document scores."""
    r1 = [rs.score(ref, p)["rouge1"].fmeasure for ref, p in zip(refs, preds)]
    rl = [rs.score(ref, p)["rougeL"].fmeasure for ref, p in zip(refs, preds)]
    mean = {"rouge1_f1": float(np.mean(r1)), "rougeL_f1": float(np.mean(rl)),
            "avg_pred_words": float(np.mean([len(p.split()) for p in preds]))}
    return mean, {"rouge1_f1": r1, "rougeL_f1": rl}


## 9. Baselines before fine-tuning

In [40]:
# What does fine-tuning add? Score the model BEFORE training (PEFT weights are still at their start values,
# so this is essentially the Phase 1 model asked to summarize), and a simple "lead" baseline that just copies the
# first words of the document. Both are used later in the analysis.
base_docs = val_docs[:N_BASE_GEN]
base_refs = [d["summary"] for d in base_docs]

base_preds = summarize(base_docs)
before_scores, _ = evaluate(base_preds, base_refs)

avg_ref_words = int(np.mean([len(r.split()) for r in base_refs]))
lead_preds = [" ".join(d["document"].split()[:avg_ref_words]) for d in base_docs]
lead_scores, _ = evaluate(lead_preds, base_refs)

print("before fine-tuning:", before_scores)
print("lead baseline     :", lead_scores)


generating:   0%|          | 0/2 [00:00<?, ?it/s]

before fine-tuning: {'rouge1_f1': 0.22089777051493412, 'rougeL_f1': 0.12299436738202801, 'avg_pred_words': 197.86666666666667}
lead baseline     : {'rouge1_f1': 0.5023930110873149, 'rougeL_f1': 0.2657527436968444, 'avg_pred_words': 756.4}


## 10. Training

### Optimizer, schedule and helpers

In [41]:
optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad],
                              lr=LR, betas=(0.9, 0.99), weight_decay=WEIGHT_DECAY)
# fp16 needs a gradient scaler on T4 GPUs (they have no bf16)
amp_dtype = torch.bfloat16 if (device == "cuda" and torch.cuda.get_device_capability()[0] >= 8) else torch.float16
scaler = torch.amp.GradScaler("cuda", enabled=(device == "cuda" and amp_dtype == torch.float16))

steps_per_epoch = len(train_x) // (MICRO_BATCH * GRAD_ACCUM)
total_steps = steps_per_epoch * EPOCHS
print(f"{steps_per_epoch} steps per epoch, {total_steps} total optimizer steps")


def lr_at(step, progress):
    """Linear warm-up, then cosine decay."""
    if step < WARMUP_STEPS:
        return LR * (step + 1) / WARMUP_STEPS
    return 0.1 * LR + 0.5 * (LR - 0.1 * LR) * (1 + math.cos(math.pi * min(1.0, progress)))


def get_batch(xs, ys, idx):
    return xs[idx].long().to(device), ys[idx].long().to(device)


@torch.no_grad()
def validation_loss(n=64):
    """Average next-token loss on the summary tokens of a few held-out documents."""
    model.eval()
    total, count = 0.0, 0
    for i in range(0, min(n, len(val_x)), MICRO_BATCH):
        x, y = get_batch(val_x, val_y, torch.arange(i, min(i + MICRO_BATCH, n, len(val_x))))
        with torch.autocast(device_type=device, dtype=amp_dtype, enabled=(device == "cuda")):
            model(x, y)
        total += model.head.main_loss.item(); count += 1
    model.train()
    return total / count


417 steps per epoch, 834 total optimizer steps


### Training loop

In [42]:
model.train()
if device == "cuda":
    torch.cuda.reset_peak_memory_stats()           # so we can report the peak GPU memory used by training
step, run_loss, run_n = 0, 0.0, 0
t_start = time.time()
stop = False
pbar = tqdm(total=total_steps, desc="fine-tuning")

for epoch in range(EPOCHS):
    perm = torch.randperm(len(train_x), generator=torch.Generator().manual_seed(SEED + epoch))
    for s in range(steps_per_epoch):
        # progress is whichever is larger: steps done, or time used (so the LR always finishes decaying)
        progress = max(step / total_steps, (time.time() - t_start) / (MAX_TRAIN_MINUTES * 60))
        if progress >= 1.0:
            stop = True; break
        lr = lr_at(step, progress)
        for g in optimizer.param_groups:
            g["lr"] = lr

        # gradient accumulation: several small batches add up to one big batch
        for m in range(GRAD_ACCUM):
            idx = perm[(s * GRAD_ACCUM + m) * MICRO_BATCH:(s * GRAD_ACCUM + m + 1) * MICRO_BATCH]
            x, y = get_batch(train_x, train_y, idx)
            with torch.autocast(device_type=device, dtype=amp_dtype, enabled=(device == "cuda")):
                _, loss = model(x, y)
            scaler.scale(loss / GRAD_ACCUM).backward()
            run_loss += model.head.main_loss.item() / GRAD_ACCUM

        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], GRAD_CLIP)
        scaler.step(optimizer); scaler.update()
        optimizer.zero_grad(set_to_none=True)
        step += 1; run_n += 1; pbar.update(1)

        if step % LOG_EVERY == 0:
            pbar.write(f"step {step}/{total_steps} | train loss {run_loss / run_n:.4f} | lr {lr:.2e} | "
                       f"{(time.time() - t_start) / 60:.1f} min")
            run_loss, run_n = 0.0, 0
        if step % EVAL_EVERY == 0:
            v = validation_loss()
            pbar.write(f"   >> val loss {v:.4f} | ppl {math.exp(v):.2f}")
    if stop:
        pbar.write("Time budget reached - stopping early."); break
pbar.close()

train_minutes = (time.time() - t_start) / 60
peak_mem_gb = torch.cuda.max_memory_allocated() / 1e9 if device == "cuda" else 0.0

# Save ONLY the trainable PEFT parameters (tiny file compared with the full model)
trainable_state = {n: p.detach().cpu() for n, p in model.named_parameters() if p.requires_grad}
torch.save({"method": METHOD, "arch": ARCH_NAME, "state": trainable_state}, PEFT_WEIGHTS_PATH)
final_val = validation_loss()
print(f"saved {len(trainable_state)} PEFT tensors -> {PEFT_WEIGHTS_PATH} | final val loss {final_val:.4f}")


fine-tuning:   0%|          | 0/834 [00:00<?, ?it/s]

step 20/834 | train loss 3.4613 | lr 2.00e-03 | 0.4 min
step 40/834 | train loss 3.4649 | lr 2.99e-03 | 0.7 min
step 60/834 | train loss 3.4709 | lr 2.97e-03 | 1.1 min
step 80/834 | train loss 3.5008 | lr 2.94e-03 | 1.5 min
step 100/834 | train loss 3.4871 | lr 2.91e-03 | 1.9 min
step 120/834 | train loss 3.4680 | lr 2.87e-03 | 2.2 min
step 140/834 | train loss 3.4722 | lr 2.82e-03 | 2.6 min
step 160/834 | train loss 3.5090 | lr 2.77e-03 | 3.0 min
step 180/834 | train loss 3.4535 | lr 2.70e-03 | 3.4 min
step 200/834 | train loss 3.4424 | lr 2.64e-03 | 3.8 min
   >> val loss 3.5277 | ppl 34.04
step 220/834 | train loss 3.4622 | lr 2.57e-03 | 4.2 min
step 240/834 | train loss 3.4673 | lr 2.49e-03 | 4.6 min
step 260/834 | train loss 3.4905 | lr 2.41e-03 | 4.9 min
step 280/834 | train loss 3.4659 | lr 2.32e-03 | 5.3 min
step 300/834 | train loss 3.4789 | lr 2.23e-03 | 5.7 min
step 320/834 | train loss 3.4936 | lr 2.14e-03 | 6.1 min
step 340/834 | train loss 3.4764 | lr 2.04e-03 | 6.4 min
s

## 11. Evaluate on held-out documents (BERTScore, ROUGE-1, ROUGE-L)

In [43]:
# Score the fine-tuned model on held-out documents with the competition metrics
eval_docs = val_docs[:N_VAL_GEN]
val_refs = [d["summary"] for d in eval_docs]

t0 = time.time()
val_preds = summarize(eval_docs)
infer_sec_per_doc = (time.time() - t0) / len(eval_docs)

scores, per_doc = evaluate(val_preds, val_refs)

results = {"arch": ARCH_NAME, "method": METHOD, **scores,
           "avg_ref_words": float(np.mean([len(r.split()) for r in val_refs])),
           "before_finetune": before_scores, "lead_baseline": lead_scores,
           # cost of the experiment
           "trainable_params_M": n_trainable / 1e6, "total_params_M": n_all / 1e6,
           "peft_file_MB": os.path.getsize(PEFT_WEIGHTS_PATH) / 1e6,
           "train_minutes": train_minutes, "peak_train_mem_GB": peak_mem_gb,
           "infer_sec_per_doc": infer_sec_per_doc, "final_val_loss": final_val,
           "settings": {"doc_len": DOC_LEN, "head_fraction": HEAD_FRACTION, "sum_len": SUM_LEN,
                        "repetition_penalty": REPETITION_PENALTY, "no_repeat_ngram": NO_REPEAT_NGRAM}}
print(json.dumps(results, indent=1))
print("\nExample summary:\n", val_preds[0])


generating:   0%|          | 0/7 [00:00<?, ?it/s]

{
 "arch": "arch4_encdec",
 "method": "soft_prompt",
 "rouge1_f1": 0.17493801533456946,
 "rougeL_f1": 0.09951022059226002,
 "avg_pred_words": 164.46,
 "avg_ref_words": 780.02,
 "before_finetune": {
  "rouge1_f1": 0.22089777051493412,
  "rougeL_f1": 0.12299436738202801,
  "avg_pred_words": 197.86666666666667
 },
 "lead_baseline": {
  "rouge1_f1": 0.5023930110873149,
  "rougeL_f1": 0.2657527436968444,
  "avg_pred_words": 756.4
 },
 "trainable_params_M": 0.012288,
 "total_params_M": 189.379968,
 "peft_file_MB": 0.050892,
 "train_minutes": 15.874211851755778,
 "peak_train_mem_GB": 2.211502592,
 "infer_sec_per_doc": 0.40544069051742554,
 "final_val_loss": 3.527391791343689,
 "settings": {
  "doc_len": 768,
  "head_fraction": 0.6,
  "sum_len": 256,
  "repetition_penalty": 1.15,
  "no_repeat_ngram": 3
 }
}

Example summary:
 The appeal is allowed. No costs. Appeal dismissed. J. (H.L. DATTU) J. (CHANDRAMAULI KR. PRASAD) NEW DELHI; JULY 23, 2011. REPORTABLE IN THE SUPREME COURT OF INDIA CIVIL A

## 12. Analysis (long documents, length, redundancy, coverage, cost)

In [44]:
# Analysis questions from the Phase 6 spec, answered with numbers computed on the held-out documents.
doc_tokens = [len(fast_encode(d["document"])) for d in eval_docs]
pred_words = np.array([len(p.split()) for p in val_preds])
ref_words = np.array([len(r.split()) for r in val_refs])
analysis = {}

# --- 1. Long documents: do documents that had to be cut score worse than documents that fit? ---
fits = np.array([n <= DOC_LEN - N_SOFT for n in doc_tokens])
analysis["long_documents"] = {"share_truncated": float(1 - fits.mean())}
for name, mask in [("fits_in_window", fits), ("truncated", ~fits)]:
    if mask.sum() >= 3:
        analysis["long_documents"][name] = {"n": int(mask.sum()),
            **{k: float(np.mean(np.array(v)[mask])) for k, v in per_doc.items()}}

# --- 2. Generation length: score for short / medium / long summaries (3 equal-size groups) ---
order = np.argsort(pred_words)
analysis["length_effect"] = {"pred_to_ref_length_ratio": float(pred_words.mean() / ref_words.mean())}
for name, idx in zip(["shortest_third", "middle_third", "longest_third"], np.array_split(order, 3)):
    analysis["length_effect"][name] = {"avg_pred_words": float(pred_words[idx].mean()),
        **{k: float(np.mean(np.array(v)[idx])) for k, v in per_doc.items()}}


# --- 3. Redundancy: how many 3-word phrases are repeated inside one summary? (0 = no repetition) ---
def repeated_trigram_rate(text):
    w = text.split()
    tri = [tuple(w[i:i + 3]) for i in range(len(w) - 2)]
    return 0.0 if not tri else 1 - len(set(tri)) / len(tri)

analysis["redundancy"] = {"repeated_trigram_rate_pred": float(np.mean([repeated_trigram_rate(p) for p in val_preds])),
                          "repeated_trigram_rate_ref": float(np.mean([repeated_trigram_rate(r) for r in val_refs]))}

# --- 4. Coverage and legal facts ---
# "facts" = numbers (years, sections, amounts) and capitalised words (names, courts, statutes).
def facts(text):
    return set(re.findall(r"\d[\d,./-]*\d|\d", text)) | set(re.findall(r"(?<![.!?] )(?<!^)\b[A-Z][a-z]{2,}\b", text))

recall, precision = [], []
for p, r, d in zip(val_preds, val_refs, eval_docs):
    fp, fr, fd = facts(p), facts(r), facts(d["document"])
    if fr:
        recall.append(len(fp & fr) / len(fr))                  # coverage: reference facts that the summary mentions
    if fp:
        precision.append(len(fp & fd) / len(fp))               # factuality proxy: summary facts that exist in the source
analysis["coverage_and_facts"] = {"fact_recall_vs_reference": float(np.mean(recall)),
                                  "fact_supported_by_source": float(np.mean(precision))}

results["analysis"] = analysis
print(json.dumps(analysis, indent=1))

# Save everything for the comparison notebook
json.dump(results, open(os.path.join(OUTPUT_DIR, "results.json"), "w"), indent=1)
json.dump([{"id": d["id"], "prediction": p, "reference": r} for d, p, r in zip(eval_docs, val_preds, val_refs)],
          open(os.path.join(OUTPUT_DIR, "val_predictions.json"), "w"), indent=1)


{
 "long_documents": {
  "share_truncated": 0.98,
  "truncated": {
   "n": 98,
   "rouge1_f1": 0.1737447758984267,
   "rougeL_f1": 0.09915396700285724
  }
 },
 "length_effect": {
  "pred_to_ref_length_ratio": 0.2108407476731366,
  "shortest_third": {
   "avg_pred_words": 45.470588235294116,
   "rouge1_f1": 0.07447390442876668,
   "rougeL_f1": 0.050674753022612776
  },
  "middle_third": {
   "avg_pred_words": 205.1818181818182,
   "rouge1_f1": 0.17103200295559254,
   "rougeL_f1": 0.09681820301472742
  },
  "longest_third": {
   "avg_pred_words": 246.33333333333334,
   "rouge1_f1": 0.2823525056164948,
   "rougeL_f1": 0.1525175683930655
  }
 },
 "redundancy": {
  "repeated_trigram_rate_pred": 0.0,
  "repeated_trigram_rate_ref": 0.07803552291500519
 },
 "coverage_and_facts": {
  "fact_recall_vs_reference": 0.1391261774239048,
  "fact_supported_by_source": 0.39220371608557103
 }
}


## 13. Test predictions and submission zip

In [45]:
# Generate one summary for every test document and write the submission (list format, one entry per ID)
test_summaries = summarize(test_docs)
predictions = [{"id": d["id"], "summary": s} for d, s in zip(test_docs, test_summaries)]
assert len(predictions) == len(test_docs)

with open(PREDICTION_PATH, "w", encoding="utf-8") as f:
    json.dump(predictions, f, ensure_ascii=False, indent=1)
with zipfile.ZipFile(SUBMISSION_ZIP, "w", zipfile.ZIP_DEFLATED) as z:
    z.write(PREDICTION_PATH, arcname="prediction.json")
print("saved", PREDICTION_PATH, "and", SUBMISSION_ZIP)
print("\nExample:\n", predictions[0]["summary"])


generating:   0%|          | 0/7 [00:00<?, ?it/s]

saved /kaggle/working/phase6_arch4_encdec_soft_prompt/prediction.json and /kaggle/working/phase6_arch4_encdec_soft_prompt/submission.zip

Example:
 J. [H.L. DATNAG] J. [R. K. RAMISW] New Delhi; April 11, 2011. REPORTABLE IN THE SUPREME COURT OF INDIA CIVIL APPELLATE JURISDICTION CIVIL APPEAL NO. 626 OF 2011 (Arising out of SLP (C) No. 31483/2005) State of Andhra Pradesh & Ors. Appellants Versus Dr. B.S. CHAUHANI AND Respondent(s) WITH CIVIL APPEAL NOS. 5601-7808 OF 2011 [Arising out of S.L.P. (C) No. 21592/2009] J U D G M E N T S. RAVINDRA BHAT, J. 1. Leave granted. 2. The appellants are aggrieved by the judgment and order dated 30th September, 2010 passed in Writ Petition No. 1014 of 2009 whereby the High Court allowed the writ petition filed by the appellants herein and quashed the order dated 31st March, 2010 passed by the District Magistrate, Jaipur (hereinafter referred to as the 'Commission') and directed the respondents to pay a sum of Rs. 1,50,000/- towards the compensation pay